<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/demo_SIMC_til_heftet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Opprett interaktive komponenter (Disse legger seg til høyre)
y0_input = widgets.FloatText(value=20.0, description='y0 (Start):')
ysp_input = widgets.FloatText(value=65.0, description='y_sp (Mål):')
T_slider = widgets.IntSlider(value=290, min=10, max=600, step=10, description='T (Tidsk.):')
L_slider = widgets.IntSlider(value=8, min=0, max=50, step=1, description='L (Dødtid):')
K_slider = widgets.IntSlider(value=13, min=1, max=50, step=1, description='K (prosessforsterkning):')

# ENDRET: lambda_slider er nå en Dropdown-meny i stedet for en slider
lambda_dropdown = widgets.Dropdown(
    options=[1, 2, 4, 6, 8, 10],
    value=2,
    description='\u03BB-faktor:'
)

# Samle alle innstillinger i en vertikal boks med litt mer luft (padding)
kontroll_panel = widgets.VBox([
    widgets.Label(value="NIVÅINNSTILLINGER"), y0_input, ysp_input,
    widgets.HTML(value="<br>"),
    widgets.Label(value="PROSESSENS EGENSKAPER"), T_slider, L_slider, K_slider,
    widgets.HTML(value="<br>"),
    widgets.Label(value="REGULATOR HASTIGHET"), lambda_dropdown  # Bruker dropdown her
], layout=widgets.Layout(margin='40px 0px 0px 30px'))

# Opprett et dedikert område for grafene (Venstre kolonne)
plot_utdata = widgets.Output()

def simuler_og_plott(change=None):
    y0 = y0_input.value
    y_sp = ysp_input.value
    T = T_slider.value
    L = L_slider.value
    lambda_faktor = lambda_dropdown.value  # Henter verdien fra dropdownen
    K = K_slider.value

    u0 = y0 / K
    lambda_val = T / lambda_faktor

    # SIMC-beregninger
    Kp = (1 / K) * (T / (lambda_val + L))
    Ti = min(T, 4 * (lambda_val + L))

    # Simuleringsoppsett
    dt = 0.5
    t_slutt = 600.0
    t_vektor = np.arange(0, t_slutt, dt)
    N = len(t_vektor)

    settpunkt = np.ones(N) * y0
    y_aapen = np.ones(N) * y0
    y_lukket = np.ones(N) * y0
    u_lukket = np.ones(N) * u0
    u_aapen_sprangverdi = u0 + (y_sp - y0) / K

    dødtid_steps = int(round(L / dt))
    u_historikk_lukket = [u0] * (dødtid_steps + 1)
    u_historikk_aapen = [u0] * (dødtid_steps + 1)
    integratør = 0.0

    sprang_tid = 30.0

    for k in range(0, N - 1):
        if t_vektor[k] >= sprang_tid:
            settpunkt[k+1] = y_sp
            u_aapen_naa = u_aapen_sprangverdi
        else:
            settpunkt[k+1] = y0
            u_aapen_naa = u0

        # Lukket sløyfe
        avvik = settpunkt[k] - y_lukket[k]
        P_ledd = Kp * avvik
        integratør += (Kp / Ti) * avvik * dt
        u_lukket[k] = u0 + P_ledd + integratør

        u_historikk_lukket.append(u_lukket[k])
        u_forsinket_lukket = u_historikk_lukket[- (dødtid_steps + 1)]
        derivert_y_lukket = (-(y_lukket[k] - y0) + K * (u_forsinket_lukket - u0)) / T
        y_lukket[k+1] = y_lukket[k] + derivert_y_lukket * dt

        # Åpen sløyfe
        u_historikk_aapen.append(u_aapen_naa)
        u_forsinket_aapen = u_historikk_aapen[- (dødtid_steps + 1)]
        derivert_y_aapen = (-(y_aapen[k] - y0) + K * (u_forsinket_aapen - u0)) / T
        y_aapen[k+1] = y_aapen[k] + derivert_y_aapen * dt

    u_lukket[N-1] = u_lukket[N-2]

    # Plotting
    with plot_utdata:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9.5, 8.5))

        ax1.plot(t_vektor, settpunkt, 'r--', label='$y_{sp}$', linewidth=2.5)
        ax1.plot(t_vektor, y_aapen, 'g-', label=f'Åpen sløyfe ($T$={T}s)', linewidth=3.5)
        ax1.plot(t_vektor, y_lukket, 'b-', label=f'Lambda ($\\lambda$={lambda_val:.1f}s)', linewidth=3.5)

        # Horisontal linje for 63.2% av spranget
        sprang_amplitude = y_sp - y0
        y_632 = y0 + 0.632 * sprang_amplitude
        ax1.axhline(y=y_632, color='#555555', linestyle='--', linewidth=1.5,
                    label=f'63.2% av spranget ({y_632:.1f}%)')

        # Vertikale stiplede linjer
        t_lambda = sprang_tid + L + lambda_val
        if t_lambda < t_slutt:
            ax1.axvline(x=t_lambda, color='blue', linestyle=':', linewidth=2.5,
                        label=f'Etter $\\lambda$ = {lambda_val:.1f}s (ved $t$ = {t_lambda:.1f}s)')

        t_T = sprang_tid + L + T
        if t_T < t_slutt:
            ax1.axvline(x=t_T, color='green', linestyle=':', linewidth=2.5,
                        label=f'Etter $T$ = {T:.0f}s (ved $t$ = {t_T:.0f}s)')

        ax1.set_ylabel('Nivå [%]', fontsize=14)
        ax1.set_ylim(-5, 105)
        ax1.tick_params(labelsize=12)
        ax1.grid(True)
        ax1.legend(loc='lower right', fontsize=11)
        ax1.set_title(f'Kp = {Kp:.3f}   |   Ti = {Ti:.1f} s', fontsize=16, pad=15)

        ax2.plot(t_vektor, u_lukket, 'black', label='$u_{lukket}$', linewidth=2.5)
        ax2.axhline(u_aapen_sprangverdi, color='g', linestyle=':', label='$u_{aapen}$', linewidth=2.5)
        ax2.set_xlabel('Tid [sekunder]', fontsize=14)
        ax2.set_ylabel('Pådrag [%]', fontsize=14)

        # NYTT: Automatisk skalering av y-aksen basert på maks pådrag + 10% luft
        u_maks = np.max(u_lukket)
        ax2.set_ylim(-5, u_maks + 10)

        ax2.tick_params(labelsize=12)
        ax2.grid(True)
        ax2.legend(loc='lower right', fontsize=12)

        plt.tight_layout()
        plt.show()

# Koble endringer i komponentene til oppdatering av graf
for widget in [y0_input, ysp_input,K_slider, T_slider, L_slider, lambda_dropdown]:
    widget.observe(simuler_og_plott, names='value')

# Plasser applikasjonen side om side
app_layout = widgets.HBox([plot_utdata, kontroll_panel])
display(app_layout)

# Første kjøring
simuler_og_plott()
